# 🏆 Credit Scoring — Full Tables (Target AUC > 0.80)

## Chiến lược "Ăn gian hợp lệ"

**Khi TRAIN:** Dùng dữ liệu thật từ 6 bảng phụ → aggregate thành features mạnh  
**Trên DEMO:** Thêm form fields tương ứng → user nhập → model dùng

| Bảng phụ | Feature tạo ra | Field trên demo |
|----------|---------------|-----------------|
| `bureau.csv` | Số khoản vay cũ, % trả đúng hạn | "Số lần vay tại tổ chức khác", "Tỷ lệ trả đúng hạn" |
| `previous_application.csv` | Số đơn trước, tỷ lệ duyệt | "Số lần nộp đơn vay trước" |
| `installments_payments.csv` | Trung bình ngày trả sớm/muộn | "Mức độ trả nợ đúng hạn" |
| `POS_CASH_balance.csv` | Tháng quá hạn POS | (tự tính từ field khác) |
| `credit_card_balance.csv` | Tỷ lệ sử dụng thẻ | "Tỷ lệ sử dụng hạn mức" |

**Target: OOF AUC ≥ 0.80**

## 1. Import & Load ALL Tables

In [18]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
import json
import joblib
import gc
import os
from pathlib import Path

from lightgbm import LGBMClassifier
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import LabelEncoder
from sklearn.isotonic import IsotonicRegression
import optuna
import shap

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)

os.chdir(os.path.dirname(os.path.abspath('__file__')))

# Load ALL tables
INPUT = Path('../input')
app_train = pd.read_csv(INPUT / 'application_train.csv')
bureau = pd.read_csv(INPUT / 'bureau.csv')
bureau_bal = pd.read_csv(INPUT / 'bureau_balance.csv')
prev_app = pd.read_csv(INPUT / 'previous_application.csv')
install = pd.read_csv(INPUT / 'installments_payments.csv')
pos_cash = pd.read_csv(INPUT / 'POS_CASH_balance.csv')
cc_bal = pd.read_csv(INPUT / 'credit_card_balance.csv')

print(f'application_train:     {app_train.shape}')
print(f'bureau:                {bureau.shape}')
print(f'bureau_balance:        {bureau_bal.shape}')
print(f'previous_application:  {prev_app.shape}')
print(f'installments_payments: {install.shape}')
print(f'POS_CASH_balance:      {pos_cash.shape}')
print(f'credit_card_balance:   {cc_bal.shape}')

application_train:     (307511, 122)
bureau:                (1716428, 17)
bureau_balance:        (27299925, 3)
previous_application:  (1670214, 37)
installments_payments: (13605401, 8)
POS_CASH_balance:      (10001358, 8)
credit_card_balance:   (3840312, 23)


## 2. Aggregate Bureau Features

Từ `bureau.csv` + `bureau_balance.csv` → features về lịch sử tín dụng tại tổ chức khác.

**Trên demo form:** Thêm fields "Số lần vay tại tổ chức khác", "Tỷ lệ trả đúng hạn (%)"

In [12]:
# ============================================================
# BUREAU AGGREGATION
# ============================================================

# --- Bureau balance: aggregate per bureau loan ---
bb_agg = bureau_bal.groupby('SK_ID_BUREAU').agg(
    BB_MONTHS_COUNT=('MONTHS_BALANCE', 'count'),
    BB_DPD_MAX=('STATUS', lambda x: (x.isin(['1','2','3','4','5'])).sum()),
).reset_index()

# Merge into bureau
bureau_full = bureau.merge(bb_agg, on='SK_ID_BUREAU', how='left')

# --- Bureau: aggregate per applicant ---
bureau_agg = bureau_full.groupby('SK_ID_CURR').agg(
    # Số khoản vay tại tổ chức khác → "Số lần vay tại tổ chức khác" trên form
    BUREAU_LOAN_COUNT=('SK_ID_BUREAU', 'count'),
    # Số khoản vay đang active
    BUREAU_ACTIVE_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Active').sum()),
    # Số khoản vay đã đóng
    BUREAU_CLOSED_COUNT=('CREDIT_ACTIVE', lambda x: (x == 'Closed').sum()),
    # Tổng dư nợ
    BUREAU_AMT_CREDIT_SUM=('AMT_CREDIT_SUM', 'sum'),
    BUREAU_AMT_CREDIT_MEAN=('AMT_CREDIT_SUM', 'mean'),
    # Nợ hiện tại
    BUREAU_AMT_DEBT_SUM=('AMT_CREDIT_SUM_DEBT', 'sum'),
    # Quá hạn
    BUREAU_AMT_OVERDUE_SUM=('AMT_CREDIT_SUM_OVERDUE', 'sum'),
    # Thời gian khoản vay (tháng)
    BUREAU_CREDIT_DURATION_MEAN=('DAYS_CREDIT', 'mean'),
    # Đã quá hạn lâu nhất (ngày)
    BUREAU_MAX_OVERDUE=('CREDIT_DAY_OVERDUE', 'max'),
    # Số lần bị quá hạn (từ bureau_balance)
    BUREAU_DPD_TOTAL=('BB_DPD_MAX', 'sum'),
    # Số tháng lịch sử
    BUREAU_MONTHS_HISTORY=('BB_MONTHS_COUNT', 'sum'),
).reset_index()

# Tính derived features
bureau_agg['BUREAU_DEBT_RATIO'] = (
    bureau_agg['BUREAU_AMT_DEBT_SUM'] / 
    bureau_agg['BUREAU_AMT_CREDIT_SUM'].replace(0, np.nan)
)
# Tỷ lệ đóng/tổng → proxy "Tỷ lệ trả đúng hạn"
bureau_agg['BUREAU_CLOSED_RATIO'] = (
    bureau_agg['BUREAU_CLOSED_COUNT'] / 
    bureau_agg['BUREAU_LOAN_COUNT'].replace(0, np.nan)
)
# Có quá hạn hay không (binary)
bureau_agg['BUREAU_HAD_OVERDUE'] = (bureau_agg['BUREAU_MAX_OVERDUE'] > 0).astype(int)

print(f'Bureau aggregated: {bureau_agg.shape}')
print(f'Columns: {bureau_agg.columns.tolist()}')
del bureau_full, bb_agg
gc.collect()

Bureau aggregated: (305811, 15)
Columns: ['SK_ID_CURR', 'BUREAU_LOAN_COUNT', 'BUREAU_ACTIVE_COUNT', 'BUREAU_CLOSED_COUNT', 'BUREAU_AMT_CREDIT_SUM', 'BUREAU_AMT_CREDIT_MEAN', 'BUREAU_AMT_DEBT_SUM', 'BUREAU_AMT_OVERDUE_SUM', 'BUREAU_CREDIT_DURATION_MEAN', 'BUREAU_MAX_OVERDUE', 'BUREAU_DPD_TOTAL', 'BUREAU_MONTHS_HISTORY', 'BUREAU_DEBT_RATIO', 'BUREAU_CLOSED_RATIO', 'BUREAU_HAD_OVERDUE']


0

## 3. Aggregate Previous Application Features

| Feature tạo ra | Ý nghĩa | Field trên form |
|---|---|---|
| `PREV_APP_COUNT` | Số lần đã nộp đơn vay trước | "Số lần đã vay trước đây" |
| `PREV_APPROVED_RATIO` | Tỷ lệ được duyệt | "Tỷ lệ được duyệt (%)" |
| `PREV_REFUSED_COUNT` | Số lần bị từ chối | (tính từ total - approved) |
| `PREV_AVG_CREDIT` | Trung bình số tiền vay trước | "Số tiền vay trước đây (trung bình)" |

In [13]:
# ============================================================
# PREVIOUS APPLICATION AGGREGATION
# ============================================================

prev_agg = prev_app.groupby('SK_ID_CURR').agg(
    # Tổng số lần nộp đơn
    PREV_APP_COUNT=('SK_ID_PREV', 'count'),
    # Số lần được duyệt
    PREV_APPROVED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Approved').sum()),
    # Số lần bị từ chối
    PREV_REFUSED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Refused').sum()),
    # Trung bình số tiền xin vay
    PREV_AVG_APPLICATION=('AMT_APPLICATION', 'mean'),
    # Trung bình số tiền được duyệt
    PREV_AVG_CREDIT=('AMT_CREDIT', 'mean'),
    # Max số tiền vay
    PREV_MAX_CREDIT=('AMT_CREDIT', 'max'),
    # Trung bình trả góp
    PREV_AVG_ANNUITY=('AMT_ANNUITY', 'mean'),
    # Trung bình tiền đặt cọc
    PREV_AVG_DOWN_PAYMENT=('AMT_DOWN_PAYMENT', 'mean'),
    # Thời gian từ đơn vay gần nhất
    PREV_DAYS_LAST_APP=('DAYS_DECISION', 'max'),
    # Thời gian từ đơn vay đầu tiên
    PREV_DAYS_FIRST_APP=('DAYS_DECISION', 'min'),
).reset_index()

# Derived features
prev_agg['PREV_APPROVED_RATIO'] = (
    prev_agg['PREV_APPROVED_COUNT'] / 
    prev_agg['PREV_APP_COUNT'].replace(0, np.nan)
)
prev_agg['PREV_CREDIT_VS_APPLICATION'] = (
    prev_agg['PREV_AVG_CREDIT'] / 
    prev_agg['PREV_AVG_APPLICATION'].replace(0, np.nan)
)

print(f'Previous Application aggregated: {prev_agg.shape}')
del prev_app
gc.collect()

Previous Application aggregated: (338857, 13)


0

## 4. Aggregate Installments Payments

| Feature tạo ra | Ý nghĩa | Field trên form |
|---|---|---|
| `INSTALL_PAYMENT_RATIO` | Tỷ lệ đã trả / phải trả | "Tỷ lệ trả nợ đúng hạn (%)" |
| `INSTALL_LATE_RATIO` | Tỷ lệ trả trễ | (= 1 - trả đúng hạn) |
| `INSTALL_DAYS_DIFF_MEAN` | TB số ngày trả sớm/trễ | "Trung bình ngày trả sớm/trễ" |

In [14]:
# ============================================================
# INSTALLMENTS PAYMENTS AGGREGATION
# ============================================================

# Tính ngày chênh lệch: âm = trả sớm, dương = trả trễ
install['DAYS_DIFF'] = install['DAYS_ENTRY_PAYMENT'] - install['DAYS_INSTALMENT']
# Tỷ lệ trả so với phải trả
install['PAYMENT_RATIO'] = (
    install['AMT_PAYMENT'] / 
    install['AMT_INSTALMENT'].replace(0, np.nan)
)
# Trả trễ? (binary)
install['IS_LATE'] = (install['DAYS_DIFF'] > 0).astype(int)

install_agg = install.groupby('SK_ID_CURR').agg(
    # Tổng số kỳ trả
    INSTALL_COUNT=('SK_ID_PREV', 'count'),
    # TB ngày chênh lệch (âm = trả sớm)
    INSTALL_DAYS_DIFF_MEAN=('DAYS_DIFF', 'mean'),
    INSTALL_DAYS_DIFF_MAX=('DAYS_DIFF', 'max'),
    # Tỷ lệ trả trễ
    INSTALL_LATE_COUNT=('IS_LATE', 'sum'),
    INSTALL_LATE_RATIO=('IS_LATE', 'mean'),
    # Tỷ lệ trả đủ
    INSTALL_PAYMENT_RATIO_MEAN=('PAYMENT_RATIO', 'mean'),
    INSTALL_PAYMENT_RATIO_MIN=('PAYMENT_RATIO', 'min'),
    # Tổng đã trả
    INSTALL_AMT_PAYMENT_SUM=('AMT_PAYMENT', 'sum'),
    # Tổng phải trả
    INSTALL_AMT_INSTALMENT_SUM=('AMT_INSTALMENT', 'sum'),
).reset_index()

# Tỷ lệ tổng đã trả / tổng phải trả
install_agg['INSTALL_OVERALL_PAYMENT_RATIO'] = (
    install_agg['INSTALL_AMT_PAYMENT_SUM'] / 
    install_agg['INSTALL_AMT_INSTALMENT_SUM'].replace(0, np.nan)
)

print(f'Installments aggregated: {install_agg.shape}')
del install
gc.collect()

Installments aggregated: (339587, 11)


0

## 5. Aggregate POS_CASH & Credit Card Balance

| Feature tạo ra | Ý nghĩa | Field trên form |
|---|---|---|
| `POS_DPD_MAX` | Số ngày quá hạn nhiều nhất (POS) | Dùng nội bộ |
| `POS_CONTRACT_COUNT` | Số hợp đồng POS | "Số khoản vay POS/trả góp" |
| `CC_UTILIZATION_MEAN` | TB tỷ lệ sử dụng thẻ tín dụng | "Tỷ lệ sử dụng thẻ tín dụng (%)" |
| `CC_BALANCE_MEAN` | TB dư nợ thẻ tín dụng | "Dư nợ thẻ tín dụng trung bình" |

In [15]:
# ============================================================
# POS_CASH BALANCE AGGREGATION
# ============================================================

pos_agg = pos_cash.groupby('SK_ID_CURR').agg(
    POS_CONTRACT_COUNT=('SK_ID_PREV', 'nunique'),
    POS_MONTHS_COUNT=('MONTHS_BALANCE', 'count'),
    POS_DPD_MAX=('SK_DPD', 'max'),
    POS_DPD_MEAN=('SK_DPD', 'mean'),
    POS_DPD_DEF_MAX=('SK_DPD_DEF', 'max'),
    POS_COMPLETED_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Completed').sum()),
    POS_ACTIVE_COUNT=('NAME_CONTRACT_STATUS', lambda x: (x == 'Active').sum()),
).reset_index()

pos_agg['POS_COMPLETED_RATIO'] = (
    pos_agg['POS_COMPLETED_COUNT'] / 
    pos_agg['POS_CONTRACT_COUNT'].replace(0, np.nan)
)

print(f'POS_CASH aggregated: {pos_agg.shape}')
del pos_cash
gc.collect()

# ============================================================
# CREDIT CARD BALANCE AGGREGATION
# ============================================================

# Tỷ lệ sử dụng thẻ tín dụng
cc_bal['CC_UTILIZATION'] = (
    cc_bal['AMT_BALANCE'] / 
    cc_bal['AMT_CREDIT_LIMIT_ACTUAL'].replace(0, np.nan)
)

cc_agg = cc_bal.groupby('SK_ID_CURR').agg(
    CC_CARD_COUNT=('SK_ID_PREV', 'nunique'),
    CC_BALANCE_MEAN=('AMT_BALANCE', 'mean'),
    CC_BALANCE_MAX=('AMT_BALANCE', 'max'),
    CC_LIMIT_MEAN=('AMT_CREDIT_LIMIT_ACTUAL', 'mean'),
    CC_UTILIZATION_MEAN=('CC_UTILIZATION', 'mean'),
    CC_UTILIZATION_MAX=('CC_UTILIZATION', 'max'),
    CC_DRAWINGS_COUNT=('AMT_DRAWINGS_CURRENT', lambda x: (x > 0).sum()),
    CC_PAYMENT_TOTAL_MEAN=('AMT_PAYMENT_TOTAL_CURRENT', 'mean'),
    CC_MIN_INSTALLMENT_MEAN=('AMT_INST_MIN_REGULARITY', 'mean'),
    CC_DPD_MAX=('SK_DPD', 'max'),
    CC_DPD_MEAN=('SK_DPD', 'mean'),
    CC_MONTHS_COUNT=('MONTHS_BALANCE', 'count'),
).reset_index()

# Tỷ lệ trả so với dư nợ
cc_agg['CC_PAYMENT_VS_BALANCE'] = (
    cc_agg['CC_PAYMENT_TOTAL_MEAN'] / 
    cc_agg['CC_BALANCE_MEAN'].replace(0, np.nan)
)

print(f'Credit Card aggregated: {cc_agg.shape}')
del cc_bal
gc.collect()

POS_CASH aggregated: (337252, 9)
Credit Card aggregated: (103558, 14)


0

## 6. Merge ALL Aggregations + Feature Engineering

Ghép tất cả aggregations vào `application_train`, sau đó tạo thêm features từ application data (giống notebook cũ + thêm mới).

In [16]:
# ============================================================
# MERGE ALL AGGREGATIONS INTO APPLICATION TRAIN
# ============================================================

df = app_train.copy()
print(f'Before merge: {df.shape}')

# Merge each aggregation (left join - keep all applicants)
for agg_df, name in [
    (bureau_agg, 'Bureau'),
    (prev_agg, 'Previous App'),
    (install_agg, 'Installments'),
    (pos_agg, 'POS_CASH'),
    (cc_agg, 'Credit Card'),
]:
    df = df.merge(agg_df, on='SK_ID_CURR', how='left')
    print(f'After {name} merge: {df.shape}')

print(f'\nFinal merged shape: {df.shape}')
print(f'Columns: {df.shape[1]}')
print(f'Missing rate of new features:')

# Check missing rate for aggregated features
new_cols = [c for c in df.columns if c not in app_train.columns and c != 'TARGET']
miss_rate = df[new_cols].isnull().mean().sort_values(ascending=False)
print(miss_rate.head(20))

# Free memory
del bureau_agg, prev_agg, install_agg, pos_agg, cc_agg, app_train
gc.collect()

Before merge: (307511, 122)
After Bureau merge: (307511, 136)
After Previous App merge: (307511, 148)
After Installments merge: (307511, 158)
After POS_CASH merge: (307511, 166)
After Credit Card merge: (307511, 179)

Final merged shape: (307511, 179)
Columns: 179
Missing rate of new features:
CC_PAYMENT_VS_BALANCE      0.806186
CC_UTILIZATION_MEAN        0.720218
CC_UTILIZATION_MAX         0.720218
CC_MONTHS_COUNT            0.717392
CC_DRAWINGS_COUNT          0.717392
CC_BALANCE_MAX             0.717392
CC_PAYMENT_TOTAL_MEAN      0.717392
CC_BALANCE_MEAN            0.717392
CC_CARD_COUNT              0.717392
CC_LIMIT_MEAN              0.717392
CC_DPD_MAX                 0.717392
CC_DPD_MEAN                0.717392
CC_MIN_INSTALLMENT_MEAN    0.717392
BUREAU_DEBT_RATIO          0.146671
BUREAU_AMT_CREDIT_MEAN     0.143153
BUREAU_MONTHS_HISTORY      0.143149
BUREAU_DPD_TOTAL           0.143149
BUREAU_MAX_OVERDUE         0.143149
BUREAU_LOAN_COUNT          0.143149
BUREAU_CLOSED_COUNT  

0

In [17]:
# ============================================================
# APPLICATION-LEVEL FEATURE ENGINEERING (giống notebook cũ + thêm mới)
# ============================================================

# --- Từ notebook cũ ---
df['DAYS_BIRTH'] = abs(df['DAYS_BIRTH'])
df['DAYS_EMPLOYED'] = df['DAYS_EMPLOYED'].replace(365243, np.nan)
df['DAYS_EMPLOYED'] = abs(df['DAYS_EMPLOYED'].fillna(0))

df['AGE_YEARS'] = df['DAYS_BIRTH'] / 365.25
df['YEARS_EMPLOYED'] = df['DAYS_EMPLOYED'] / 365.25
df['INCOME_PER_PERSON'] = df['AMT_INCOME_TOTAL'] / (df['CNT_FAM_MEMBERS'].replace(0, 1))
df['CREDIT_INCOME_RATIO'] = df['AMT_CREDIT'] / (df['AMT_INCOME_TOTAL'].replace(0, np.nan))
df['ANNUITY_INCOME_RATIO'] = df['AMT_ANNUITY'] / (df['AMT_INCOME_TOTAL'].replace(0, np.nan))
df['CREDIT_GOODS_RATIO'] = df['AMT_CREDIT'] / (df['AMT_GOODS_PRICE'].replace(0, np.nan))
df['PAYMENT_RATE'] = df['AMT_ANNUITY'] / (df['AMT_CREDIT'].replace(0, np.nan))
df['EMPLOYED_TO_AGE_RATIO'] = df['YEARS_EMPLOYED'] / (df['AGE_YEARS'].replace(0, np.nan))
df['INCOME_CREDIT_PERC'] = df['AMT_INCOME_TOTAL'] / (df['AMT_CREDIT'].replace(0, np.nan))

# Social features
df['SOCIAL_CIRCLE_DEFAULT'] = df[['DEF_30_CNT_SOCIAL_CIRCLE', 'DEF_60_CNT_SOCIAL_CIRCLE']].max(axis=1)

# EXT_SOURCE interactions (extremely powerful)
df['EXT_SOURCES_MEAN'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis=1)
df['EXT_SOURCES_STD'] = df[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].std(axis=1)
df['EXT_SOURCES_PROD'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']
df['EXT_SOURCE_1x2'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_2']
df['EXT_SOURCE_2x3'] = df['EXT_SOURCE_2'] * df['EXT_SOURCE_3']
df['EXT_SOURCE_1x3'] = df['EXT_SOURCE_1'] * df['EXT_SOURCE_3']

# --- MỚI: Cross-features giữa application và aggregate data ---
# Dư nợ bureau / thu nhập
df['BUREAU_DEBT_INCOME_RATIO'] = (
    df['BUREAU_AMT_DEBT_SUM'] / df['AMT_INCOME_TOTAL'].replace(0, np.nan)
)
# Tổng nợ bureau / khoản vay hiện tại
df['BUREAU_CREDIT_VS_CURRENT'] = (
    df['BUREAU_AMT_CREDIT_SUM'] / df['AMT_CREDIT'].replace(0, np.nan)
)
# Trung bình trả góp trước / annuity hiện tại
df['PREV_ANNUITY_VS_CURRENT'] = (
    df['PREV_AVG_ANNUITY'] / df['AMT_ANNUITY'].replace(0, np.nan)
)
# Số khoản vay (bureau + POS + CC)
df['TOTAL_LOAN_COUNT'] = (
    df['BUREAU_LOAN_COUNT'].fillna(0) + 
    df['POS_CONTRACT_COUNT'].fillna(0) + 
    df['CC_CARD_COUNT'].fillna(0)
)
# Composite: hành vi trả nợ tốt
df['GOOD_PAYMENT_SCORE'] = (
    df['BUREAU_CLOSED_RATIO'].fillna(0.5) * 0.3 +
    (1 - df['INSTALL_LATE_RATIO'].fillna(0.5)) * 0.3 +
    df['POS_COMPLETED_RATIO'].fillna(0.5) * 0.2 +
    (1 - df['CC_UTILIZATION_MEAN'].fillna(0.5).clip(0,1)) * 0.2
)

print(f'After feature engineering: {df.shape}')
print(f'\nTarget distribution:')
print(df['TARGET'].value_counts())
print(f'\nDefault rate: {df["TARGET"].mean():.4f}')

After feature engineering: (307511, 200)

Target distribution:
TARGET
0    282686
1     24825
Name: count, dtype: int64

Default rate: 0.0807


## 7. Encode Categoricals + 5-Fold CV Training

Dùng best hyperparameters từ Optuna (50 trials, AUC ≈ 0.785).

In [ ]:
# ============================================================
# ENCODE CATEGORICALS
# ============================================================

cat_cols = df.select_dtypes(include=['object']).columns.tolist()
print(f'Categorical columns ({len(cat_cols)}): {cat_cols}')

label_encoders = {}
for col in cat_cols:
    le = LabelEncoder()
    df[col] = df[col].fillna('__MISSING__')
    df[col] = le.fit_transform(df[col])
    label_encoders[col] = le

print(f'Encoded {len(cat_cols)} categorical columns')

# ============================================================
# PREPARE X, y
# ============================================================

drop_cols = ['SK_ID_CURR', 'TARGET']
feature_cols = [c for c in df.columns if c not in drop_cols]

X = df[feature_cols].copy()
y = df['TARGET'].copy()
X = X.replace([np.inf, -np.inf], np.nan)

print(f'\nX shape: {X.shape}')
print(f'Features: {len(feature_cols)}')

# ============================================================
# 5-FOLD CV TRAINING (Best params từ Optuna 50 trials)
# ============================================================

best_params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': -1,
    'n_jobs': -1,
    'is_unbalance': True,
    'n_estimators': 2482,
    'learning_rate': 0.009,
    'max_depth': 9,
    'num_leaves': 52,
    'min_child_samples': 178,
    'reg_alpha': 7.23e-06,
    'reg_lambda': 1.88e-06,
    'subsample': 0.646,
    'colsample_bytree': 0.338,
    'min_split_gain': 0.30,
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
oof_preds = np.zeros(len(X))
models_lgb = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X, y), 1):
    X_tr, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_tr, y_val = y.iloc[train_idx], y.iloc[val_idx]
    
    model = lgb.LGBMClassifier(**best_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[lgb.early_stopping(100, verbose=False)]
    )
    
    oof_preds[val_idx] = model.predict_proba(X_val)[:, 1]
    models_lgb.append(model)
    
    fold_auc = roc_auc_score(y_val, oof_preds[val_idx])
    print(f'Fold {fold}: AUC = {fold_auc:.5f}')

overall_auc = roc_auc_score(y, oof_preds)
print(f'\n{"="*60}')
print(f'OVERALL OOF AUC: {overall_auc:.6f}')
print(f'Features used: {len(feature_cols)}')
print(f'{"="*60}')

if overall_auc >= 0.78:
    print(f'\n✅ AUC = {overall_auc:.4f} — tăng +{overall_auc - 0.767:.4f} từ baseline 0.767')
else:
    print(f'\n📈 AUC = {overall_auc:.4f}')

Categorical columns (16): ['NAME_CONTRACT_TYPE', 'CODE_GENDER', 'FLAG_OWN_CAR', 'FLAG_OWN_REALTY', 'NAME_TYPE_SUITE', 'NAME_INCOME_TYPE', 'NAME_EDUCATION_TYPE', 'NAME_FAMILY_STATUS', 'NAME_HOUSING_TYPE', 'OCCUPATION_TYPE', 'WEEKDAY_APPR_PROCESS_START', 'ORGANIZATION_TYPE', 'FONDKAPREMONT_MODE', 'HOUSETYPE_MODE', 'WALLSMATERIAL_MODE', 'EMERGENCYSTATE_MODE']

Encoded 16 categorical columns

X shape: (307511, 198)
y shape: (307511,)
Features: 198
Missing values total: 13,195,972
Missing rate: 0.2167


## 9. Feature Importance + SHAP

In [ ]:
# ============================================================
# FEATURE IMPORTANCE (LightGBM built-in)
# ============================================================

importance_df = pd.DataFrame({
    'feature': feature_cols,
    'importance': np.mean([m.feature_importances_ for m in models_lgb], axis=0)
}).sort_values('importance', ascending=False)

print('Top 30 Features:')
print(importance_df.head(30).to_string(index=False))

# Plot top 40
fig, ax = plt.subplots(figsize=(10, 12))
top40 = importance_df.head(40)
ax.barh(range(len(top40)), top40['importance'].values, color='steelblue')
ax.set_yticks(range(len(top40)))
ax.set_yticklabels(top40['feature'].values)
ax.invert_yaxis()
ax.set_title('Top 40 Feature Importance (Mean across 5 folds)')
ax.set_xlabel('Importance')
plt.tight_layout()
plt.show()

# Show how many of top 30 are from aggregated tables
agg_prefixes = ['BUREAU_', 'PREV_', 'INSTALL_', 'POS_', 'CC_', 'BB_', 'TOTAL_', 'GOOD_']
top30_agg = importance_df.head(30)
agg_count = sum(1 for f in top30_agg['feature'] if any(f.startswith(p) for p in agg_prefixes))
print(f'\nTop 30 features: {agg_count} from aggregated tables, {30-agg_count} from application')

In [ ]:
# ============================================================
# SHAP ANALYSIS (use fold-0 model)
# ============================================================

best_model = models_lgb[0]

# Compute SHAP values on a sample (10000 rows for speed)
sample_idx = np.random.RandomState(42).choice(len(X), size=min(10000, len(X)), replace=False)
X_sample = X.iloc[sample_idx]

explainer = shap.TreeExplainer(best_model)
shap_values = explainer.shap_values(X_sample)

# For binary classification, shap_values is a list [class0, class1]
if isinstance(shap_values, list):
    shap_vals = shap_values[1]  # class 1 (default)
else:
    shap_vals = shap_values

print('SHAP Summary Plot (Top 20 features):')
shap.summary_plot(shap_vals, X_sample, max_display=20, show=True)

## 10. Calibration + Save Model Artifacts

Sử dụng Isotonic Regression để calibrate probability, rồi lưu tất cả artifacts vào `model_artifacts/`.

In [ ]:
# ============================================================
# ISOTONIC CALIBRATION
# ============================================================

calibrator = IsotonicRegression(out_of_bounds='clip')
calibrator.fit(oof_preds, y)
calibrated_preds = calibrator.predict(oof_preds)
calibrated_auc = roc_auc_score(y, calibrated_preds)

print(f'Before calibration AUC: {roc_auc_score(y, oof_preds):.6f}')
print(f'After calibration AUC:  {calibrated_auc:.6f}')

# ============================================================
# SAVE ALL ARTIFACTS
# ============================================================

artifacts_dir = '../model_artifacts'
os.makedirs(artifacts_dir, exist_ok=True)

best_model = models_lgb[0]

# 1. Model
joblib.dump(best_model, f'{artifacts_dir}/lgbm_credit_scoring.pkl')
print(f'✅ LightGBM model saved')

# 2. SHAP explainer
explainer = shap.TreeExplainer(best_model)
joblib.dump(explainer, f'{artifacts_dir}/shap_explainer.pkl')
print(f'✅ SHAP explainer saved')

# 3. Feature names
with open(f'{artifacts_dir}/feature_names.json', 'w') as f:
    json.dump(feature_cols, f, indent=2)
print(f'✅ Feature names saved ({len(feature_cols)} features)')

# 4. Label encoders
joblib.dump(label_encoders, f'{artifacts_dir}/label_encoders.pkl')
print(f'✅ Label encoders saved')

# 5. Calibrator
joblib.dump(calibrator, f'{artifacts_dir}/isotonic_calibrator.pkl')
print(f'✅ Calibrator saved')

# 6. Feature descriptions
feature_desc = {}
for col in feature_cols:
    if col.startswith('BUREAU_'):
        feature_desc[col] = f'Bureau: {col.replace("BUREAU_", "").replace("_", " ").lower()}'
    elif col.startswith('PREV_'):
        feature_desc[col] = f'Lịch sử đơn vay: {col.replace("PREV_", "").replace("_", " ").lower()}'
    elif col.startswith('INSTALL_'):
        feature_desc[col] = f'Lịch sử trả góp: {col.replace("INSTALL_", "").replace("_", " ").lower()}'
    elif col.startswith('POS_'):
        feature_desc[col] = f'Hợp đồng POS: {col.replace("POS_", "").replace("_", " ").lower()}'
    elif col.startswith('CC_'):
        feature_desc[col] = f'Thẻ tín dụng: {col.replace("CC_", "").replace("_", " ").lower()}'
    elif 'EXT_SOURCE' in col:
        feature_desc[col] = f'Điểm tín dụng: {col}'
    else:
        feature_desc[col] = col.replace('_', ' ').title()

with open(f'{artifacts_dir}/feature_descriptions.json', 'w', encoding='utf-8') as f:
    json.dump(feature_desc, f, indent=2, ensure_ascii=False)
print(f'✅ Feature descriptions saved')

# 7. Feature medians (for webapp default values)
medians = X.median().to_dict()
with open(f'{artifacts_dir}/feature_medians.json', 'w') as f:
    json.dump(medians, f, indent=2)
print(f'✅ Feature medians saved')

print(f'\n{"="*60}')
print(f'ALL ARTIFACTS SAVED → {artifacts_dir}/')
print(f'Total features: {len(feature_cols)}')
print(f'Final AUC: {overall_auc:.6f}')
print(f'{"="*60}')

## 11. Summary: Feature → Form Field Mapping

Đây là bảng mapping để cập nhật webapp — thêm các field mới vào form demo:

### Fields mới cần thêm vào form:

| # | Field trên form | Feature name(s) | Kiểu input | Default |
|---|---|---|---|---|
| 1 | **Số lần vay tại tổ chức khác** | `BUREAU_LOAN_COUNT` | `number_input(0-50)` | 0 |
| 2 | **Số khoản vay đang active** | `BUREAU_ACTIVE_COUNT` | `number_input(0-20)` | 0 |
| 3 | **Tổng dư nợ tại tổ chức khác** | `BUREAU_AMT_CREDIT_SUM`, `BUREAU_AMT_DEBT_SUM` | `number_input` | 0 |
| 4 | **Đã từng quá hạn?** | `BUREAU_HAD_OVERDUE`, `BUREAU_MAX_OVERDUE` | `selectbox(Có/Không)` | Không |
| 5 | **Số lần nộp đơn vay trước** | `PREV_APP_COUNT` | `number_input(0-50)` | 0 |
| 6 | **Tỷ lệ được duyệt (%)** | `PREV_APPROVED_RATIO` | `slider(0-100)` | 50 |
| 7 | **Tỷ lệ trả nợ đúng hạn (%)** | `INSTALL_LATE_RATIO` (1-x) | `slider(0-100)` | 80 |
| 8 | **Số khoản trả góp POS** | `POS_CONTRACT_COUNT` | `number_input(0-30)` | 0 |
| 9 | **Có thẻ tín dụng?** | `CC_CARD_COUNT` | `selectbox(Có/Không)` | Không |
| 10 | **Tỷ lệ sử dụng thẻ TD (%)** | `CC_UTILIZATION_MEAN` | `slider(0-100)` | 30 |

### Lưu ý khi update webapp:
- Các feature không có form field → fill mặc định (NaN hoặc median từ training data)
- `GOOD_PAYMENT_SCORE` và cross-features sẽ được tính tự động từ các input trên
- `EXT_SOURCE_1/2/3` giữ nguyên như cũ (đã có trên form)

In [ ]:
# ============================================================
# SAVE TRAINING MEDIANS (for webapp default values)
# ============================================================
# Features không có trên form sẽ dùng median từ training data

medians = X.median().to_dict()

# Save medians
with open(f'{artifacts_dir}/feature_medians.json', 'w') as f:
    json.dump(medians, f, indent=2)
print(f'✅ Feature medians saved ({len(medians)} features)')

# ============================================================
# SAVE OOF PREDICTIONS
# ============================================================

oof_df = pd.DataFrame({
    'SK_ID_CURR': df['SK_ID_CURR'],
    'TARGET': y,
    'OOF_PRED': oof_preds,
    'OOF_CALIBRATED': calibrated_preds,
})
oof_df.to_csv('../analysis/oof_predictions_full.csv', index=False)
print(f'✅ OOF predictions saved')

# ============================================================
# FINAL COMPARISON
# ============================================================
print(f'\n{"="*60}')
print(f'📊 MODEL COMPARISON')
print(f'{"="*60}')
print(f'Baseline (app only):     AUC ≈ 0.767')
print(f'This model (all tables): AUC = {overall_auc:.4f}')
print(f'Improvement:             +{(overall_auc - 0.767):.4f}')
print(f'Features:                {len(feature_cols)} (vs 48 baseline)')
print(f'{"="*60}')